# Deep Learning 009 — Why a Hidden Layer Helps

Lesson 007 proved one perceptron cannot do XOR. This notebook shows what a second layer
buys, first **by hand** — placing two sigmoid units and combining them deliberately —
and then by training, where we count how often training actually finds the solution.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.neural_network import MLPClassifier

def sigmoid(z):
    return 1 / (1 + np.exp(-np.clip(z, -60, 60)))

G = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], float)
XOR = np.array([0, 1, 1, 0])

## By hand: XOR is OR **and** NAND

`x1 XOR x2` is true exactly when *at least one* input is on **and** *not both* are. So
build one unit that computes OR, one that computes NAND, and AND their outputs. Every
weight below is chosen by hand — nothing is trained.

In [ ]:
# hidden unit 1: OR   (fires unless both inputs are 0)
# hidden unit 2: NAND (fires unless both inputs are 1)
W1 = np.array([[20., -20.],
               [20., -20.]])      # columns: [OR, NAND]
b1 = np.array([-10.,  30.])

# output unit: AND of the two hidden units
W2 = np.array([20., 20.])
b2 = -30.

H = sigmoid(G @ W1 + b1)
out = sigmoid(H @ W2 + b2)

print(f"{'x1':>3}{'x2':>4}{'OR':>7}{'NAND':>7}{'out':>7}{'target':>8}")
for i in range(4):
    print(f'{G[i,0]:>3.0f}{G[i,1]:>4.0f}{H[i,0]:>7.2f}{H[i,1]:>7.2f}'
          f'{out[i]:>7.2f}{XOR[i]:>8d}')
assert ((out >= 0.5).astype(int) == XOR).all()
print('\nexact XOR, with zero training')

**That is the entire argument for depth.** One layer draws one line. Two lines carve the
plane into regions, and a second layer chooses which combination of regions counts as
"on". Neither line alone separates XOR; their conjunction does.

Look at what the hidden layer did to the geometry — it moved the four points somewhere a
single line *can* split them.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(9, 3.6))
ax[0].scatter(G[XOR == 0, 0], G[XOR == 0, 1], marker='o', s=110, label='0')
ax[0].scatter(G[XOR == 1, 0], G[XOR == 1, 1], marker='^', s=110, label='1')
xs = np.linspace(-0.4, 1.4, 20)
for j, name in enumerate(['OR', 'NAND']):
    ax[0].plot(xs, -(W1[0, j] / W1[1, j]) * xs - b1[j] / W1[1, j], lw=1.4, label=name)
ax[0].set(title='input space: two hand-placed lines', xlim=(-0.4, 1.4), ylim=(-0.4, 1.4))
ax[0].legend(fontsize=8)

ax[1].scatter(H[XOR == 0, 0], H[XOR == 0, 1], marker='o', s=110, label='0')
ax[1].scatter(H[XOR == 1, 0], H[XOR == 1, 1], marker='^', s=110, label='1')
ax[1].plot([0, 1], [1, 0], 'k--', lw=1, label='now one line works')
ax[1].set(xlabel='OR output', ylabel='NAND output',
          title='hidden space: linearly separable')
ax[1].legend(fontsize=8); plt.tight_layout(); plt.show()

## By training: how often does it actually find that solution?

The solution exists. That does not mean gradient descent reliably reaches it. Count
across 100 random seeds — this is a real measurement, and the number is lower than most
people expect.

In [ ]:
def success_rate(hidden, seeds=100, max_iter=6000):
    wins = 0
    for s in range(seeds):
        m = MLPClassifier(hidden_layer_sizes=(hidden,), activation='logistic',
                          solver='lbfgs', max_iter=max_iter,
                          random_state=s).fit(G, XOR)
        wins += int((m.predict(G) == XOR).all())
    return wins / seeds

print(f"{'hidden units':>13}{'converged':>12}")
for h in (1, 2, 3, 4, 8, 16):
    print(f'{h:>13}{success_rate(h):>12.0%}')

**Two hidden units are enough in principle and unreliable in practice.** The extra units
do not add expressive power for XOR — they add *paths to the solution*, so random
initialisation is more likely to start somewhere that descends to it.

This is the first appearance of a theme that runs through the whole track: **capacity
and trainability are different things.** A network that *can* represent the answer is
not the same as a network that *will* find it.

## Exercises

1. Re-run `success_rate` with `activation='relu'`. Does the convergence rate change?
2. Set `solver='sgd'` with `learning_rate_init=0.1`. Compare the rates — and explain the
   gap in terms of lesson 006's loss surface.
3. Take one *failed* seed with 2 hidden units, print its learned weights, and work out
   what function it settled on instead.
4. Build XOR by hand a second way, using AND and OR instead of OR and NAND.